# RAG agêntico

Um RAG simples busca uma vez e responde com o que veio, esteja o trecho certo ou não. Quando a pergunta chega em linguagem do dia a dia, a busca traz trechos vizinhos que não respondem. RAG agêntico é colocar decisões em volta da busca: avaliar se os trechos servem, reescrever a pergunta quando não servem, recorrer a outra fonte quando o documento não tem a resposta e escolher a fonte antes de buscar.

Este notebook monta essas decisões como nós e arestas de um grafo, seguindo o tutorial de RAG agêntico da documentação do LangGraph, sobre o Regimento Geral da UFRN.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph pdfplumber
# !pip install -q langchain-groq langchain-google-genai

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
import json
import os
import urllib.request
from pathlib import Path
from typing import Literal
from urllib.parse import quote

import pandas as pd
from IPython.display import Image
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)
embeddings = init_embeddings("openai:text-embedding-3-small")

# Outros modelos de embedding. Ao trocar, apague a pasta chroma_regimento, porque o tamanho do vetor muda.
# embeddings = init_embeddings("openai:openai/text-embedding-3-small", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], check_embedding_ctx_length=False)
# from agentkit import Embeddings
# embeddings = Embeddings("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

## O índice do regimento

O documento é o Regimento Geral da UFRN, cortado em trechos de 1000 caracteres, cada um com a página do PDF. Se a pasta `chroma_regimento` já existe, a célula só a reabre; senão, baixa o PDF e indexa.

In [ ]:
URL = "https://www.ufrn.br/resources/documentos/regimentos/RegimentoGeral.pdf"
FILE = Path("regimento.pdf")

if not FILE.exists():
    request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        FILE.write_bytes(response.read())

vector_store = Chroma(
    collection_name="regimento",
    embedding_function=embeddings,
    persist_directory="./chroma_regimento",
    collection_metadata={"hnsw:space": "cosine"},
)

if not vector_store.get()["ids"]:
    docs = PDFPlumberLoader(str(FILE)).load()
    for doc in docs:
        doc.metadata["page"] += 1
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(docs)
    vector_store.add_documents(documents=chunks, ids=[f"chunk-{i}" for i in range(len(chunks))])

print("trechos indexados:", len(vector_store.get()["ids"]))

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


def format_docs(documents: list) -> str:
    """Junta os trechos em um texto só, com a página na frente de cada um."""
    return "\n\n".join(f"[página {doc.metadata['page']}] {doc.page_content}" for doc in documents)


@tool
def search_regiment(query: str) -> str:
    """Busca trechos do Regimento Geral da UFRN relacionados à consulta."""
    return format_docs(retriever.invoke(query))

## Onde o RAG simples falha

O RAG simples é o retriever embrulhado em ferramenta, com o `create_agent` em volta. A pergunta abaixo está escrita como um aluno escreveria; o regimento fala em deixar de comparecer às atividades e usa "falta" para infração disciplinar. Antes de rodar, tente prever: o agente acha a resposta?

In [ ]:
SYSTEM = """# Papel
Você responde dúvidas de alunos sobre a UFRN.

# Regras
Para perguntas sobre o regimento, use a ferramenta e termine a resposta com a página, no formato (página 68).
Para o resto, responda direto.
Seja breve: no máximo três frases."""

rag_agent = create_agent(model, tools=[search_regiment], system_prompt=SYSTEM)

In [ ]:
result = rag_agent.invoke({"messages": [HumanMessage("Quantas faltas posso ter numa matéria?")]})

print("caminho  :", " -> ".join(message.type for message in result["messages"]))
print("consultas:", [call["args"]["query"] for message in result["messages"] if message.type == "ai" for call in message.tool_calls])
print("resposta :", result["messages"][-1].content)

In [ ]:
print(search_regiment.invoke({"query": "Quantas faltas posso ter numa matéria?"}))

Os três trechos falam de falta como infração. A resposta está no art. 115, na página 74, que reprova quem deixa de comparecer a 25% ou mais das atividades.

Rode o agente algumas vezes e o caminho muda: às vezes ele se contenta com esses trechos, às vezes busca de novo com outras palavras e acha o artigo. Nada confere se os trechos respondem à pergunta. As próximas seções acrescentam essa decisão e as outras.

## Avaliação dos trechos

O primeiro passo é saber se a busca deu certo. A distância de cosseno não serve, porque não existe um limiar que valha para toda pergunta. A alternativa é pedir ao modelo que julgue: ele recebe a pergunta e os trechos e responde sim ou não, em saída estruturada.

In [ ]:
GRADE_PROMPT = """# Papel
Você avalia se trechos recuperados de um documento respondem a uma pergunta.

# Regras
Responda 'sim' só se algum trecho traz a informação que a pergunta pede, mesmo com outras palavras.
Responda 'nao' se os trechos só compartilham palavras ou o tema geral com a pergunta.

# Pergunta
<pergunta>
{question}
</pergunta>

# Trechos
<trechos>
{context}
</trechos>"""


class GradeDocuments(BaseModel):
    """Veredito binário sobre a relevância dos trechos."""

    binary_score: Literal["sim", "nao"] = Field(description="'sim' se os trechos respondem à pergunta, 'nao' se não")


grader = model.with_structured_output(GradeDocuments)

In [ ]:
question = "Quantas faltas posso ter numa matéria?"

for query in [question, "Qual a frequência mínima exigida para aprovação em disciplina?"]:
    context = search_regiment.invoke({"query": query})
    verdict = grader.invoke(GRADE_PROMPT.format(context=context, question=question))
    print(f"{verdict.binary_score:3s} | busca por {query!r}")

Os dois vereditos julgam a mesma pergunta do aluno; muda só a busca. Os trechos da pergunta coloquial foram reprovados, e os da versão no vocabulário do regimento, aprovados. O avaliador não melhora a busca, só diz se ela funcionou, e é esse sinal que vai decidir o próximo passo do grafo.

### Exercício 1

Escreva três perguntas suas sobre o regimento: uma formal, uma coloquial e uma cujo assunto não está no documento. Para cada uma, recupere os trechos com o `search_regiment` e passe pelo `grader`. Leia os trechos e diga se concorda com os três vereditos.

In [ ]:
# Seu código aqui

## Reescrita da pergunta

Quando os trechos são reprovados, dá para tentar de novo com outra consulta. O modelo conhece o vocabulário do aluno e o do regimento, então pode reescrever a pergunta nos termos do documento. Os trechos reprovados entram no prompt para ele entender por que a busca errou.

In [ ]:
REWRITE_PROMPT = """# Papel
Você reescreve perguntas para a busca no Regimento Geral da UFRN.

# Contexto
A busca com a pergunta abaixo não trouxe a resposta, e os trechos que vieram estão no fim.

# Regras
Leia os trechos e entenda por que eles não servem.
Reescreva a pergunta com os termos formais que o regimento usaria para o assunto certo.
Responda só com a pergunta reescrita.

# Pergunta
<pergunta>
{question}
</pergunta>

# Trechos que vieram
<trechos>
{context}
</trechos>"""

question = "Quantas faltas posso ter numa matéria?"
rejected = search_regiment.invoke({"query": question})
rewritten = model.invoke(REWRITE_PROMPT.format(question=question, context=rejected)).content

print(rewritten)
print([doc.metadata["page"] for doc in retriever.invoke(rewritten)])

Compare as páginas com as da busca original. A reescrita nem sempre acerta de primeira, porque aposta em um vocabulário, e só a avaliação diz se a aposta deu certo. Por isso, no grafo, reescrita e avaliação formam um ciclo.

### O grafo

O grafo segue o tutorial do LangGraph. O `generate_query_or_respond` chama o modelo com a ferramenta ligada, e ele decide se busca ou responde. O `retrieve` executa a busca. A aresta que sai dele, `grade_documents`, chama o avaliador e escolhe entre `generate_answer` e `rewrite_question`. A pergunta reescrita volta para o começo.

O estado ganha uma chave, `rewrites`, que conta as reescritas e garante o fim do ciclo quando a resposta não está no documento.

In [ ]:
class RAGState(MessagesState):
    rewrites: int

In [ ]:
MAX_REWRITES = 2

RAG_SYSTEM = """# Papel
Você responde perguntas sobre a UFRN consultando o Regimento Geral com a ferramenta de busca.

# Regras
Toda pergunta que peça informação passa pela busca, mesmo que você ache que sabe a resposta.
Se a última mensagem do usuário for uma pergunta reescrita, busque de novo com ela.
Para cumprimentos e conversa, responda direto."""

model_with_tools = model.bind_tools([search_regiment])


def generate_query_or_respond(state: RAGState) -> dict:
    """Chama o modelo com a ferramenta de busca ligada; ele decide se busca ou responde."""
    return {"messages": [model_with_tools.invoke([SystemMessage(RAG_SYSTEM)] + state["messages"])]}


def grade_documents(state: RAGState) -> Literal["generate_answer", "rewrite_question"]:
    """Avalia os trechos da última busca contra a pergunta original."""
    question, context = state["messages"][0].content, state["messages"][-1].content
    verdict = grader.invoke(GRADE_PROMPT.format(context=context, question=question))
    if verdict.binary_score == "sim" or state.get("rewrites", 0) >= MAX_REWRITES:
        return "generate_answer"
    return "rewrite_question"

In [ ]:
GENERATE_PROMPT = """# Papel
Você responde perguntas sobre a UFRN a partir de trechos de documentos.

# Regras
Responda usando apenas os trechos do contexto.
Cite a fonte entre parênteses, como ela aparece no contexto.
Seja breve: no máximo três frases.
Se o contexto não bastar, diga que não encontrou e não cite fonte.

# Contexto
<contexto>
{context}
</contexto>

# Pergunta
<pergunta>
{question}
</pergunta>"""


def rewrite_question(state: RAGState) -> dict:
    """Reescreve a pergunta original a partir dos trechos que a avaliação reprovou."""
    question, context = state["messages"][0].content, state["messages"][-1].content
    rewritten = model.invoke(REWRITE_PROMPT.format(question=question, context=context)).content
    return {"messages": [HumanMessage(rewritten)], "rewrites": state.get("rewrites", 0) + 1}


def generate_answer(state: RAGState) -> dict:
    """Gera a resposta com a pergunta original e os trechos da última busca."""
    question, context = state["messages"][0].content, state["messages"][-1].content
    return {"messages": [model.invoke(GENERATE_PROMPT.format(context=context, question=question))]}

In [ ]:
rag_builder = StateGraph(RAGState)

# nós
rag_builder.add_node("generate_query_or_respond", generate_query_or_respond)
rag_builder.add_node("retrieve", ToolNode([search_regiment]))
rag_builder.add_node("rewrite_question", rewrite_question)
rag_builder.add_node("generate_answer", generate_answer)

# arestas
rag_builder.add_edge(START, "generate_query_or_respond")
rag_builder.add_conditional_edges("generate_query_or_respond", tools_condition, {"tools": "retrieve", END: END})
rag_builder.add_conditional_edges("retrieve", grade_documents)
rag_builder.add_edge("rewrite_question", "generate_query_or_respond")
rag_builder.add_edge("generate_answer", END)

rag_graph = rag_builder.compile()
Image(rag_graph.get_graph().draw_mermaid_png())

O `grade_documents` é uma aresta, e não um nó: escolhe o destino e não escreve no estado, por isso não aparece como caixa no desenho. A função `show` roda o grafo, imprime cada nó que executa e, no fim, a resposta.

In [ ]:
def show(graph, question: str) -> None:
    """Roda o grafo, imprime cada nó executado e, no fim, a resposta."""
    for step in graph.stream({"messages": [HumanMessage(question)]}, stream_mode="updates"):
        for node, update in step.items():
            print(node)
    print()
    print(update["messages"][-1].content)

In [ ]:
show(rag_graph, "Quantas faltas posso ter numa matéria?")

A primeira busca trouxe os trechos disciplinares, a avaliação os reprovou e o grafo passou pelo `rewrite_question`. O ciclo se repete até os trechos trazerem o art. 115. A resposta é gerada para a pergunta original, e não para a reescrita, que existe só para a busca.

A próxima pergunta é sobre a UFRN, mas o ano de criação não está no regimento. Antes de rodar, tente prever quantas buscas o grafo faz.

In [ ]:
show(rag_graph, "Em que ano a UFRN foi criada?")

Foram três buscas, a original e uma por reescrita, até o `rewrites` chegar ao `MAX_REWRITES` e o grafo responder com o que tinha. Reescrever não resolve o que o documento não tem.

### Exercício 2

Rode o `rag_graph` com `MAX_REWRITES` igual a 0 e igual a 4, com a pergunta das faltas e com a do ano de criação da UFRN, e compare os caminhos. O `MAX_REWRITES` é lido na hora em que a aresta roda, então basta mudar a variável e chamar o `show` de novo. Qual valor você usaria para o regimento, e quanto cada reescrita a mais custa em chamadas ao modelo?

In [ ]:
# Seu código aqui

## Correção com outra fonte

Quando o documento não tem a resposta, o RAG corretivo troca de fonte: se a avaliação reprovar mesmo depois das reescritas, o grafo busca na web. A fonte aqui é a Wikipédia em português, pela API pública, que não exige chave.

In [ ]:
def fetch(url: str) -> dict:
    """Faz um GET e devolve o JSON da resposta."""
    request = urllib.request.Request(url, headers={"User-Agent": "ia-agentica-aula/1.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


@tool
def search_wikipedia(query: str) -> str:
    """Busca na Wikipédia em português e devolve a introdução dos dois artigos mais relevantes."""
    pages = fetch(
        "https://pt.wikipedia.org/w/api.php?action=query&format=json&generator=search"
        f"&gsrsearch={quote(query)}&gsrlimit=2&prop=extracts&exintro=1&explaintext=1"
    )["query"]["pages"]
    return "\n\n".join(f"[Wikipédia: {page['title']}] {page['extract']}" for page in pages.values())

In [ ]:
print(search_wikipedia.invoke({"query": "Em que ano a UFRN foi criada?"}))

A ferramenta devolve texto no mesmo formato do `search_regiment`, com a fonte entre colchetes, e por isso o mesmo `generate_answer` serve para as duas. O nó `web_search` busca a pergunta original e acrescenta o resultado ao histórico como uma `ToolMessage`; o `generate_answer` lê a última mensagem, como antes.

In [ ]:
def web_search(state: RAGState) -> dict:
    """Busca a pergunta original na Wikipédia."""
    result = search_wikipedia.invoke({"query": state["messages"][0].content})
    return {"messages": [ToolMessage(result, tool_call_id="web_search")]}


def grade_or_search(state: RAGState) -> Literal["generate_answer", "rewrite_question", "web_search"]:
    """Como o grade_documents, mas recorre à Wikipédia quando as reescritas acabam."""
    question, context = state["messages"][0].content, state["messages"][-1].content
    if grader.invoke(GRADE_PROMPT.format(context=context, question=question)).binary_score == "sim":
        return "generate_answer"
    return "web_search" if state.get("rewrites", 0) >= MAX_REWRITES else "rewrite_question"

In [ ]:
corrective_builder = StateGraph(RAGState)

# nós
corrective_builder.add_node("generate_query_or_respond", generate_query_or_respond)
corrective_builder.add_node("retrieve", ToolNode([search_regiment]))
corrective_builder.add_node("rewrite_question", rewrite_question)
corrective_builder.add_node("web_search", web_search)
corrective_builder.add_node("generate_answer", generate_answer)

# arestas
corrective_builder.add_edge(START, "generate_query_or_respond")
corrective_builder.add_conditional_edges("generate_query_or_respond", tools_condition, {"tools": "retrieve", END: END})
corrective_builder.add_conditional_edges("retrieve", grade_or_search)
corrective_builder.add_edge("rewrite_question", "generate_query_or_respond")
corrective_builder.add_edge("web_search", "generate_answer")
corrective_builder.add_edge("generate_answer", END)

corrective_graph = corrective_builder.compile()
Image(corrective_graph.get_graph().draw_mermaid_png())

In [ ]:
show(corrective_graph, "Em que ano a UFRN foi criada?")

O caminho repete o do grafo anterior até a última avaliação, e ali o grafo foi à Wikipédia; a resposta cita o artigo de onde saiu o ano. O regimento vem antes porque é a fonte de autoridade para as regras da universidade, e a web entra só como correção.

### Exercício 3

O `web_search` busca com a pergunta original, e a busca da Wikipédia funciona melhor com palavras-chave do que com perguntas inteiras. Escreva uma versão do nó que peça ao modelo de duas a quatro palavras-chave antes de buscar, monte o grafo com ela e compare os títulos dos artigos que voltam para a pergunta do ano de criação da UFRN e para uma pergunta sua fora do regimento.

In [ ]:
# Seu código aqui

## Roteamento

O grafo corretivo só chega à Wikipédia depois de três buscas, três avaliações e duas reescritas, mesmo quando a pergunta deixa claro que o regimento não tem a resposta. O roteamento decide na entrada: um nó classifica a pergunta e manda para o regimento, para a Wikipédia ou, se for conversa, para uma resposta direta.

In [ ]:
ROUTE_PROMPT = """# Papel
Você é o roteador de um assistente que responde dúvidas sobre a UFRN.

# Regras
Classifique a pergunta do usuário em exatamente uma de três fontes: regimento, web ou conversa.
Regimento cobre normas, órgãos, cargos e procedimentos acadêmicos e administrativos da UFRN.
Web cobre fatos gerais, como história, datas, pessoas e lugares, mesmo que sejam sobre a UFRN.
Conversa cobre cumprimentos e mensagens que não pedem informação."""


class RouteQuery(BaseModel):
    """Fonte escolhida para responder à pergunta."""

    source: Literal["regimento", "web", "conversa"]


router = model.with_structured_output(RouteQuery)

ROUTES = {"regimento": "generate_query_or_respond", "web": "web_search", "conversa": "respond"}


def route_question(state: RAGState) -> Command[Literal["generate_query_or_respond", "web_search", "respond"]]:
    """Classifica a pergunta e segue direto para o nó da fonte escolhida."""
    route = router.invoke([SystemMessage(ROUTE_PROMPT)] + state["messages"])
    return Command(goto=ROUTES[route.source])


def respond(state: RAGState) -> dict:
    """Responde à conversa sem ferramenta."""
    return {"messages": [model.invoke(state["messages"])]}

O `route_question` usa `Command`: o nó escolhe o destino no `goto`, e a anotação `Command[Literal[...]]` diz ao grafo quais nós podem vir depois dele. O resto é o grafo corretivo.

In [ ]:
routed_builder = StateGraph(RAGState)

# nós
routed_builder.add_node("route_question", route_question)
routed_builder.add_node("respond", respond)
routed_builder.add_node("generate_query_or_respond", generate_query_or_respond)
routed_builder.add_node("retrieve", ToolNode([search_regiment]))
routed_builder.add_node("rewrite_question", rewrite_question)
routed_builder.add_node("web_search", web_search)
routed_builder.add_node("generate_answer", generate_answer)

# arestas
routed_builder.add_edge(START, "route_question")
routed_builder.add_edge("respond", END)
routed_builder.add_conditional_edges("generate_query_or_respond", tools_condition, {"tools": "retrieve", END: END})
routed_builder.add_conditional_edges("retrieve", grade_or_search)
routed_builder.add_edge("rewrite_question", "generate_query_or_respond")
routed_builder.add_edge("web_search", "generate_answer")
routed_builder.add_edge("generate_answer", END)

routed_graph = routed_builder.compile()
Image(routed_graph.get_graph().draw_mermaid_png())

In [ ]:
show(routed_graph, "Em que ano a UFRN foi criada?")

In [ ]:
show(routed_graph, "Bom dia, tudo bem?")

A pergunta sobre o ano de criação foi direto para a Wikipédia, sem passar pelo regimento, e a saudação foi respondida sem busca. O custo é uma chamada a mais em toda pergunta, a do roteador. E um roteador que erra manda uma pergunta sobre normas para a web, onde nenhuma avaliação a traz de volta.

### Exercício 4

Escreva oito perguntas: três sobre normas do regimento, três de fatos gerais sobre a UFRN ou sobre educação e duas de conversa, anotando para cada uma a fonte que você espera. Passe todas pelo `router` e monte um `DataFrame` com a pergunta, a fonte esperada e a escolhida. Onde ele errou, ajuste o `ROUTE_PROMPT` e rode de novo.

In [ ]:
# Seu código aqui